# Client Boto 3

In [1]:
import os
import io 
import boto3
import json
from pprint import pprint
import pandas as pd
from tqdm.notebook import tqdm  #  Barre Jupyter native (bleue)
# ou : from tqdm.autonotebook import tqdm  # auto console/notebook

import sys
import statistics

import html
from bs4 import BeautifulSoup
import re
from typing import Dict, List, Optional, Any

import pyarrow.parquet as pq
import pyarrow.json as paj
import pyarrow as pa

from tqdm import tqdm
import time

endpoint = os.environ["S3_ENDPOINT_URL"]
bucket = os.environ["S3_BUCKET"]

s3_boto = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)

def read_wttj_parquet_file_to_df(storage, prefix: str) -> pd.DataFrame:
    """
    Reads Welcome To The Jungle data from the Silver layer (Parquet) with parallelization.
    
    Args:
        storage: Storage instance
        prefix: S3/local prefix for Silver WTTJ data
        
    Returns:
        DataFrame with normalized columns
    """
    logger.info(f"📂 Reading Welcome To The Jungle Silver: {prefix}")
    
    keys = list(storage.list_keys(prefix))
    parquet_keys = [k for k in keys if k.endswith('.parquet')]
    logger.info(f"   Found {len(parquet_keys)} Parquet files")
    
    if len(parquet_keys) == 0:
        logger.warning(f"⚠️ No Parquet files found in {prefix}")
        return pd.DataFrame()
    
    # Parallelization for Parquet files
    # With a pool of 50 connections, we can go up to 40 workers
    max_workers = min(int(os.getenv("WTTJ_READ_WORKERS", "40")), len(parquet_keys))
    dfs = []
    
    if len(parquet_keys) > 1:
        logger.info(f"   Using {max_workers} workers for parallelization")
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(process_wttj_jsonl_file, storage, key): key for key in parquet_keys}
            
            for i, future in enumerate(as_completed(futures), 1):
                try:
                    df = future.result()
                    if not df.empty:
                        dfs.append(df)
                    logger.info(f"   WTTJ Progress: {i}/{len(parquet_keys)} files")
                except Exception as e:
                    key = futures[future]
                    logger.error(f"   ❌ Error processing {key}: {e}")
    else:
        # If only one file, no need for parallelization
        try:
            df = storage.read_parquet(parquet_keys[0])
            dfs.append(df)
        except Exception as e:
            logger.error(f"   ❌ Error reading {parquet_keys[0]}: {e}")
    
    if not dfs:
        logger.warning("⚠️ No WTTJ data found")
        return pd.DataFrame()
    
    df = pd.concat(dfs, ignore_index=True)
    logger.info(f"✅ Welcome To The Jungle: {len(df):,} offers loaded")
    return df    




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

ModuleNotFoundError: No module named 'src'